# Cours SQL — Formation Beobank · SAS → Python · Orsys

**Ce notebook est à la fois le support du formateur ET le cahier du participant.**

- Progression complète façon **W3Schools SQL** : chaque instruction (`SELECT`, `WHERE`,
  `JOIN`, `GROUP BY`...) a sa propre section, avec définition, syntaxe, **pourquoi on
  l'utilise**, et un exemple exécuté sur les vraies tables Beobank.
- **Orientation Python + SGBD d'entreprise (Vertica)** : le SQL lui-même est standard
  (les requêtes ne changent pas), mais ce cours insiste sur **comment piloter SQL
  depuis Python** — connexion, curseur, exécution, récupération des résultats — avec
  les repères précis pour **Vertica** (le SGBD analytique cible chez Beobank), en plus
  du bac à sable pédagogique **SQLite** utilisé pour exécuter les exemples ici.
- **Chaque requête est commentée clause par clause** (`-- commentaire SQL`), et chaque
  fonction Python utilisée autour (`pd.read_sql()`, `conn.execute()`, `cursor.fetchall()`...)
  est expliquée.
- Chaque section se termine par un exercice `✏️` + une correction `✅`.
- Chaque section a un encart `🔗 Pour aller plus loin` (référence W3Schools + doc SQLite/Vertica).

> **Prérequis :** `../cours_numpy_pandas/Cours_Pandas.ipynb` (on réutilise `pd.read_sql()`
> pour récupérer un résultat SQL sous forme de DataFrame). Ce cours va plus loin que le
> Module 8 du `Jour3_Formateur.ipynb` (qui ne couvre que CREATE/INSERT/UPDATE) : il part
> de zéro et couvre l'intégralité du langage SQL de base.
>
> **Pourquoi SQLite et pas directement Vertica dans ce notebook ?** Aucun serveur
> Vertica n'est accessible dans cet environnement de formation. SQLite permet
> d'exécuter du vrai SQL, immédiatement, sans installation ni accès réseau. La Section 3bis
> explique précisément ce qui est **identique** et ce qui **diffère** entre les deux,
> pour que ces exemples restent directement transposables à un cluster Vertica réel.

## Sommaire

1. [Qu'est-ce que SQL et pourquoi l'utiliser ?](#s1) · 2. [Syntaxe et commentaires SQL](#s2) ·
3. [Setup — charger les données Beobank](#s3) · **3bis. [Manipuler SQL depuis Python : DB-API et Vertica](#s3b)** ·
4. [SELECT](#s4) · 5. [SELECT DISTINCT](#s5) ·
6. [WHERE](#s6) · 7. [ORDER BY](#s7) · 8. [AND, OR, NOT](#s8) · 9. [INSERT INTO](#s9) ·
10. [Valeurs NULL](#s10) · 11. [UPDATE](#s11) · 12. [DELETE](#s12) · 13. [LIMIT (SELECT TOP)](#s13) ·
14. [Fonctions d'agrégation (COUNT, SUM, AVG, MIN, MAX)](#s14) · 15. [LIKE et jokers](#s15) ·
16. [IN](#s16) · 17. [BETWEEN](#s17) · 18. [Alias (AS)](#s18) · 19. [INNER JOIN](#s19) ·
20. [LEFT JOIN](#s20) · 21. [RIGHT JOIN et FULL JOIN](#s21) · 22. [SELF JOIN](#s22) ·
23. [UNION et UNION ALL](#s23) · 24. [GROUP BY](#s24) · 25. [HAVING](#s25) ·
26. [CASE](#s26) · 27. [Fonctions NULL (COALESCE)](#s27) · 28. [EXISTS et sous-requêtes](#s28) ·
29. [CREATE TABLE](#s29) · 30. [Mini-projet final](#s30) · 31. [Aide-mémoire et liens](#s31)

## Glossaire des abréviations du dataset Beobank

| Abréviation | Signification |
|---|---|
| `IDT_AC` | Identifiant du compte |
| `IDT_PI` | Identifiant du tiers / de la personne |
| `SLD_CTR` | Solde du contrat |
| `SLD_DSP` | Solde disponible |
| `MNT_INI` | Montant initial |
| `COD_ECV_CTR` | Code état civil du contrat (1=ouvert, 4=clôturé, 6=résilié...) |
| `COD_DEV` | Code devise |
| `COD_TYP_TIE` | Code type de tiers (1=Personne Physique, 2=Personne Morale) |
| `DAT_OUV_CTR` | Date d'ouverture du contrat |

<a id="s1"></a>
## 1. Qu'est-ce que SQL et pourquoi l'utiliser ?

**SQL** (*Structured Query Language*) est LE langage standard pour interroger des
bases de données relationnelles (tables avec des lignes et des colonnes, reliées
entre elles par des clés). Ce n'est pas un langage de programmation généraliste comme
Python : c'est un langage **déclaratif**, on décrit **quel résultat** on veut, pas
**comment** l'obtenir (le moteur de base de données se charge du "comment").

| | Pandas (déjà vu) | SQL |
|---|---|---|
| Où tournent les calculs | en mémoire, dans le processus Python | dans le moteur de base de données |
| Usage typique | analyse exploratoire, data science | interroger une base de production, du reporting standardisé |
| Équivalent `ctr[ctr["SLD_CTR"]>0]` | filtre Pandas | `SELECT * FROM CTR WHERE SLD_CTR > 0` |

**Pourquoi c'est indispensable en entreprise ?** Beobank, comme toute banque, stocke
ses données dans des bases relationnelles. SQL est LE langage que tout système
(reporting, applications métier, entrepôts de données) sait parler. Python + Pandas
sert à analyser ; SQL sert à INTERROGER LA SOURCE — les deux se combinent en permanence,
exactement comme dans ce notebook : on écrit du SQL, on récupère le résultat dans
un DataFrame Pandas avec `pd.read_sql()`.

🔗 **Pour aller plus loin :**
[W3Schools — SQL Intro](https://www.w3schools.com/sql/sql_intro.asp) ·
[SQLite — À propos](https://www.sqlite.org/about.html)

<a id="s2"></a>
## 2. Syntaxe générale et commentaires SQL

Quelques règles de syntaxe SQL à connaître avant de commencer :

- Les **mots-clés** (`SELECT`, `FROM`, `WHERE`...) s'écrivent traditionnellement en
  MAJUSCULES (convention, pas une obligation — SQL n'est pas sensible à la casse
  pour ses mots-clés) pour les distinguer des noms de colonnes/tables.
- Une instruction SQL se termine par un **point-virgule** `;` (facultatif dans un
  script à une seule requête, mais bonne pratique).
- Les noms de colonnes/tables SONT sensibles à la casse selon le moteur (SQLite est
  plutôt tolérant, mais gardez une orthographe cohérente).
- **Commentaires** : `-- commentaire sur une ligne` ou `/* commentaire sur plusieurs lignes */`.

In [ ]:
# Exemple de requête commentée -- la structure qu'on va détailler section par section
exemple_sql = """
    -- Ceci est un commentaire SQL : ignoré par le moteur, utile pour documenter
    SELECT IDT_AC, SLD_CTR      -- les colonnes qu'on veut récupérer
    FROM CTR                     -- la table source
    WHERE SLD_CTR > 0            -- le filtre appliqué
    ORDER BY SLD_CTR DESC        -- le tri du résultat
    LIMIT 5                      -- ne garder que les 5 premières lignes
"""
print(exemple_sql)

🔗 **Pour aller plus loin :**
[W3Schools — SQL Syntax](https://www.w3schools.com/sql/sql_syntax.asp) ·
[W3Schools — SQL Comments](https://www.w3schools.com/sql/sql_comments.asp)

<a id="s3"></a>
## 3. Setup — charger les données Beobank dans SQLite

On charge les 5 tables CSV dans une base **SQLite en mémoire** (`:memory:` — elle
existe le temps de la session, puis disparaît). C'est la même technique qu'au Jour 3.

In [ ]:
import sqlite3                       # module standard Python : base de données SQLite légère
import pandas as pd                  # pour charger les CSV et récupérer les résultats SQL
import numpy as np                   # pour simuler les colonnes manquantes de TXN_X_CTR
from pathlib import Path             # gestion de chemins de fichiers indépendante de l'OS

DATA = Path("../data")                            # dossier contenant les CSV Beobank
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")   # paramètres de lecture communs (vus au Jour 2)

# --- Chargement des 5 tables avec Pandas (le plus simple pour ensuite les charger en SQL) ---
ctr     = pd.read_csv(DATA / "CTR.csv",       **PARAMS)   # 200 contrats
tie     = pd.read_csv(DATA / "TIE.csv",       **PARAMS)   # 100 clients
tie_adr = pd.read_csv(DATA / "TIE_ADR.csv",   **PARAMS)   # 100 adresses
txc     = pd.read_csv(DATA / "TIE_X_CTR.csv", **PARAMS)   # 200 liens client-contrat
txn     = pd.read_csv(DATA / "TXN_X_CTR.csv", **PARAMS)   # 1260 transactions

# TXN_X_CTR.csv n'a pas de montant/date de mouvement exploitable -> simulation pédagogique
# (même technique et même graine qu'aux Jours 2 et 3, pour rester cohérent avec le reste de la formation)
rng = np.random.default_rng(42)                            # générateur aléatoire reproductible (graine 42)
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])     # date de mouvement simulée
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)  # montant simulé (moyenne 250€, écart-type 400€)

# --- Connexion SQLite en mémoire ---
conn = sqlite3.connect(":memory:")     # sqlite3.connect(":memory:") : base de données temporaire, en RAM

# .to_sql(nom_table, connexion, ...) : écrit un DataFrame Pandas comme une VRAIE table SQL
for nom, df in [("CTR", ctr), ("TIE", tie), ("TIE_ADR", tie_adr),
                ("TIE_X_CTR", txc), ("TXN_X_CTR", txn)]:
    df.to_sql(nom, conn, if_exists="replace", index=False)   # if_exists="replace" : écrase si déjà présente

print("Tables chargées dans SQLite :", ["CTR", "TIE", "TIE_ADR", "TIE_X_CTR", "TXN_X_CTR"])

🔗 **Pour aller plus loin :**
[Python — module sqlite3 (documentation officielle)](https://docs.python.org/3/library/sqlite3.html) ·
[Pandas — pandas.DataFrame.to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)

<a id="s3b"></a>
## 3bis. Manipuler SQL depuis Python — DB-API 2.0 et Vertica

**Tous les modules Python qui parlent à une base de données SQL suivent la MÊME
interface standard : la DB-API 2.0 (PEP 249).** `sqlite3` (module standard, utilisé
dans ce notebook), `vertica_python` (pilote officiel Vertica), `psycopg2`
(PostgreSQL), `pyodbc`... TOUS exposent le même schéma d'utilisation :

| Étape | Méthode | Rôle |
|---|---|---|
| 1. Se connecter | `connect(...)` | ouvre une connexion réseau vers le serveur (ou un fichier/mémoire pour SQLite) |
| 2. Créer un curseur | `connexion.cursor()` | l'objet qui exécute les requêtes et lit les résultats |
| 3. Exécuter | `curseur.execute(sql, parametres)` | envoie la requête SQL au serveur |
| 4. Récupérer | `curseur.fetchone()` / `.fetchmany(n)` / `.fetchall()` | lit les lignes du résultat |
| 5. Valider (si écriture) | `connexion.commit()` | rend permanents les INSERT/UPDATE/DELETE |
| 6. Fermer | `curseur.close()` / `connexion.close()` | libère les ressources |

**Conséquence pratique :** apprendre ce pattern UNE fois avec SQLite (gratuit, pas de
serveur à installer) permet de l'appliquer directement à Vertica, PostgreSQL, ou tout
autre SGBD — seuls changent les paramètres de connexion et quelques détails de syntaxe SQL.

### 3bis.1 Le pattern bas niveau : cursor.execute() + fetch

In [ ]:
import sqlite3

# .cursor() : crée un curseur SUR la connexion déjà ouverte (conn vient de la Section 3 Setup)
curseur = conn.cursor()

# .execute(sql, parametres) : envoie la requête -- le "?" est un PLACEHOLDER,
# remplacé en sécurité par la valeur du tuple (jamais de f-string dans du SQL : risque d'injection SQL)
curseur.execute("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR > ?", (5000,))

# .description : métadonnées des colonnes du résultat -- utile pour retrouver les noms de colonnes
noms_colonnes = [col[0] for col in curseur.description]
print("Colonnes :", noms_colonnes)

# .fetchmany(n) : récupère AU PLUS n lignes (pratique pour ne pas charger un résultat énorme d'un coup)
premieres_lignes = curseur.fetchmany(3)
for ligne in premieres_lignes:            # chaque ligne est un TUPLE de valeurs, dans l'ordre des colonnes
    print(ligne)

# .fetchall() : récupère TOUT LE RESTE du résultat (ce qui n'a pas encore été lu)
reste = curseur.fetchall()
print(f"Lignes restantes récupérées avec fetchall() : {len(reste)}")

curseur.close()    # libère le curseur (bonne pratique, surtout sur un vrai serveur distant)

💡 **Pourquoi utiliser `cursor.execute()` plutôt que `pd.read_sql()` partout ?**
`pd.read_sql()` (utilisé dans tout le reste de ce notebook) est plus court et
renvoie directement un DataFrame — parfait pour l'ANALYSE. Le pattern bas niveau
`cursor.execute()` / `fetchall()` est ce qu'on utilise pour des opérations
d'ÉCRITURE (INSERT/UPDATE/DELETE, vues aux Sections 9, 11, 12) ou quand on veut
traiter les lignes UNE PAR UNE sans tout charger en mémoire (`fetchmany()` par
lots, utile sur une très grosse table Vertica).

### 3bis.2 Se connecter à Vertica avec `vertica_python`

In [ ]:
# vertica_python : le pilote Python OFFICIEL pour Vertica -- respecte la même DB-API 2.0
# Installation dans un projet réel :  pip install vertica-python

try:
    import vertica_python   # si le pilote est installé dans l'environnement...

    # Paramètres de connexion -- à adapter à VOTRE cluster Vertica.
    # RÈGLE DE SÉCURITÉ : ne jamais écrire un mot de passe en clair dans le code.
    # En pratique : variables d'environnement (os.environ), ou un fichier de config non versionné.
    import os
    parametres_connexion = {
        "host":     "vertica.beobank.local",           # adresse du serveur (ou de l'équilibreur de charge)
        "port":     5433,                                # port par défaut de Vertica
        "user":     os.environ.get("VERTICA_USER", "moi"),
        "password": os.environ.get("VERTICA_PWD", ""),    # lu depuis une variable d'environnement, jamais en dur
        "database": "beobank_dwh",                         # nom de la base analytique
        "autocommit": False,                                 # comme sqlite3 : écriture validée par commit() explicite
        "connection_timeout": 5,                              # évite d'attendre indéfiniment si le serveur ne répond pas
    }

    connexion_vertica = vertica_python.connect(**parametres_connexion)   # connect() : ouvre la connexion réseau
    curseur_vertica = connexion_vertica.cursor()                          # même méthode .cursor() qu'avec sqlite3

    # EXACTEMENT le même pattern que sqlite3 -- SEULE différence : le symbole de placeholder est %s, pas ?
    curseur_vertica.execute("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR > %s", (5000,))
    for ligne in curseur_vertica.fetchmany(5):
        print(ligne)

    curseur_vertica.close()
    connexion_vertica.close()

except ModuleNotFoundError:
    # Dans CET environnement de formation, vertica_python n'est pas installé et aucun cluster
    # Vertica n'est accessible -- c'est normal. Le code ci-dessus reste la RÉFÉRENCE exacte
    # à réutiliser tel quel une fois en poste, contre un vrai serveur Vertica Beobank.
    print("vertica_python n'est pas installé ici -- ce code sert de modèle de référence pour un vrai cluster Vertica.")
except Exception as erreur:
    # En conditions réelles : identifiants invalides, réseau indisponible, base inexistante...
    print(f"Connexion Vertica impossible dans cet environnement : {type(erreur).__name__}")

### 3bis.3 `pd.read_sql()` fonctionne aussi avec une connexion Vertica

In [ ]:
# pd.read_sql(requete, connexion) accepte N'IMPORTE QUELLE connexion DB-API 2.0 --
# sqlite3.Connection ICI, mais vertica_python.Connection fonctionnerait EXACTEMENT pareil
# une fois connecté à un vrai serveur :
#
#     df = pd.read_sql("SELECT * FROM CTR WHERE SLD_CTR > 5000", connexion_vertica)
#
# Aucune ligne de code Pandas ne change entre SQLite et Vertica : seule la CONNEXION change.
# C'est tout l'intérêt du standard DB-API 2.0 : le code d'ANALYSE est portable d'un SGBD à l'autre.

apercu = pd.read_sql("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR > 5000 LIMIT 5", conn)
print(apercu)

### 3bis.4 SQLite (ce notebook) vs Vertica (cible Beobank) : différences à connaître

| | SQLite (bac à sable ici) | Vertica (SGBD cible Beobank) |
|---|---|---|
| Placeholder de paramètre | `?` | `%s` |
| Limiter le nombre de lignes | `LIMIT n` | `LIMIT n` (identique) ou `TOP n` |
| Recherche texte insensible à la casse | `LIKE` (déjà insensible sur l'ASCII) | `LIKE` (sensible à la casse) **ou** `ILIKE` (insensible, dédié) |
| Typage des colonnes | dynamique, peu strict | strict : `VARCHAR`, `INT`, `NUMERIC`, `TIMESTAMP`... déclarés à la création |
| Chargement en masse de données | `INSERT` ligne par ligne, ou `.to_sql()` Pandas | commande **`COPY`** : bien plus rapide, conçue pour charger des millions de lignes (fichier CSV/Parquet directement dans une table) |
| Organisation des tables | une seule base, pas de schéma | tables organisées par **schéma** (`beobank.ctr`, `staging.ctr`...) |
| Architecture | fichier local ou mémoire, mono-utilisateur | **MPP colonnaire** distribué sur plusieurs nœuds — conçu pour l'analytique sur de très gros volumes |
| Fonctions de fenêtre (`OVER`, `PARTITION BY`) | supportées depuis SQLite 3.25+ | supportées nativement, très optimisées (point fort de Vertica, voir Jour 3) |

**Le SQL "standard" (SELECT, WHERE, JOIN, GROUP BY...) que vous venez d'apprendre et
allez pratiquer dans tout le reste de ce notebook fonctionne QUASIMENT À L'IDENTIQUE
sur Vertica.** Les quelques différences ci-dessus sont signalées, avec un encart
"💡 Vertica", dans les sections concernées plus loin (INSERT, LIMIT, LIKE, CREATE TABLE).

### ✏️ Exercice 3bis — Traduire une requête paramétrée vers Vertica

In [ ]:
# Voici une requête écrite pour sqlite3 (avec le placeholder "?") :
requete_sqlite = "SELECT IDT_AC, SLD_CTR FROM CTR WHERE COD_DEV = ? AND SLD_CTR > ?"
parametres = ("EUR", 10000)

# TODO 1 : réécrivez cette même requête en syntaxe Vertica (placeholder %s au lieu de ?)
#          dans une variable "requete_vertica"
# TODO 2 : exécutez la version sqlite3 ci-dessus avec curseur.execute(requete_sqlite, parametres)
#          et affichez le résultat avec fetchall()

# --- votre code ---

### ✅ Correction Exercice 3bis

In [ ]:
# 1. Traduction vers la syntaxe Vertica -- seul le symbole de placeholder change
requete_vertica = "SELECT IDT_AC, SLD_CTR FROM CTR WHERE COD_DEV = %s AND SLD_CTR > %s"
print("Version Vertica :", requete_vertica)

# 2. Exécution avec sqlite3 (le pattern est rigoureusement identique avec vertica_python)
requete_sqlite = "SELECT IDT_AC, SLD_CTR FROM CTR WHERE COD_DEV = ? AND SLD_CTR > ?"
parametres = ("EUR", 10000)

curseur = conn.cursor()
curseur.execute(requete_sqlite, parametres)
for ligne in curseur.fetchall():
    print(ligne)
curseur.close()

### 3bis.5 Bonnes pratiques Python + SQL (valables sur SQLite comme sur Vertica)

- **Toujours des requêtes paramétrées** (`?` ou `%s`), jamais de f-string/`.format()`
  pour injecter une valeur dans du SQL — c'est la porte ouverte à l'**injection SQL**,
  une faille de sécurité classique.
- **Fermer ce qu'on ouvre** : curseurs et connexions, idéalement avec `with` (gestionnaire
  de contexte) pour une fermeture automatique même en cas d'erreur.
- **`commit()` après chaque écriture** (INSERT/UPDATE/DELETE) — sans lui, rien n'est
  définitivement enregistré (voir Sections 9, 11, 12).
- **Ne jamais coder un mot de passe en dur** : variable d'environnement, gestionnaire
  de secrets d'entreprise, jamais dans un notebook partagé.

In [ ]:
import sqlite3

# with ... as ... : gestionnaire de contexte -- ferme AUTOMATIQUEMENT la ressource
# à la sortie du bloc, même si une erreur survient à l'intérieur (try/finally implicite)
with sqlite3.connect(":memory:") as demo_connexion:      # ici une base jetable, juste pour la démonstration
    demo_curseur = demo_connexion.cursor()
    demo_curseur.execute("CREATE TABLE test (x INTEGER)")
    demo_curseur.execute("INSERT INTO test VALUES (?)", (42,))
    demo_connexion.commit()                                # valide l'écriture
    demo_curseur.execute("SELECT * FROM test")
    print(demo_curseur.fetchall())
    demo_curseur.close()
# ici, demo_connexion est automatiquement fermée -- pas besoin d'appeler .close() nous-mêmes

🔗 **Pour aller plus loin :**
[PEP 249 — Python Database API Specification v2.0](https://peps.python.org/pep-0249/) ·
[vertica-python — dépôt officiel GitHub](https://github.com/vertica/vertica-python) ·
[Documentation Vertica — SQL Reference](https://docs.vertica.com/latest/en/sql-reference/) ·
[Documentation Vertica — COPY statement](https://docs.vertica.com/latest/en/sql-reference/statements/copy/) ·
[Python — module sqlite3 (documentation officielle)](https://docs.python.org/3/library/sqlite3.html)

<a id="s4"></a>
## 4. SELECT

`SELECT` choisit les **colonnes** à afficher, `FROM` indique la **table** source.
`SELECT *` récupère TOUTES les colonnes (pratique pour explorer, à éviter en
production sur une grosse table pour ne pas transférer inutilement des données).

In [ ]:
# pd.read_sql(requete_sql, connexion) : exécute la requête et renvoie le résultat en DataFrame
requete = """
    SELECT IDT_AC, SLD_CTR, COD_DEV   -- on choisit 3 colonnes précises
    FROM CTR                          -- table source
"""
resultat = pd.read_sql(requete, conn)
print(resultat.head())

# SELECT * -- toutes les colonnes, pratique pour un premier coup d'oeil
apercu = pd.read_sql("SELECT * FROM CTR", conn)
print(f"\n{apercu.shape[1]} colonnes récupérées avec SELECT *")

### ✏️ Exercice 1 — Sélectionner des colonnes de TIE

In [ ]:
# TODO 1 : écrire un SELECT sur IDT_PI, COD_TYP_TIE, COD_LNG_CTR depuis la table TIE
# TODO 2 : exécuter avec pd.read_sql() et afficher .head()

# --- votre code ---

### ✅ Correction Exercice 1

In [ ]:
requete = "SELECT IDT_PI, COD_TYP_TIE, COD_LNG_CTR FROM TIE"
resultat = pd.read_sql(requete, conn)
print(resultat.head())

🔗 **Pour aller plus loin :**
[W3Schools — SQL SELECT](https://www.w3schools.com/sql/sql_select.asp)

<a id="s5"></a>
## 5. SELECT DISTINCT

`DISTINCT` élimine les **doublons** du résultat — équivalent SQL de `.unique()` en Pandas.

In [ ]:
requete = """
    SELECT DISTINCT COD_DEV    -- DISTINCT : ne garder chaque valeur qu'UNE seule fois
    FROM CTR
"""
devises = pd.read_sql(requete, conn)
print(devises)

# Sur PLUSIEURS colonnes : DISTINCT s'applique à la COMBINAISON des colonnes
combinaisons = pd.read_sql("SELECT DISTINCT COD_ECV_CTR, COD_DEV FROM CTR", conn)
print(f"\n{len(combinaisons)} combinaisons distinctes (statut, devise)")

### ✏️ Exercice 2 — Langues distinctes

In [ ]:
# TODO : écrire un SELECT DISTINCT sur COD_LNG_CTR depuis TIE

# --- votre code ---

### ✅ Correction Exercice 2

In [ ]:
langues = pd.read_sql("SELECT DISTINCT COD_LNG_CTR FROM TIE", conn)
print(langues)

🔗 **Pour aller plus loin :**
[W3Schools — SQL SELECT DISTINCT](https://www.w3schools.com/sql/sql_distinct.asp)

<a id="s6"></a>
## 6. WHERE

`WHERE` filtre les **lignes** selon une condition — équivalent SQL du filtre booléen
Pandas `df[condition]`. Opérateurs de comparaison : `=`, `!=` (ou `<>`), `>`, `<`, `>=`, `<=`.

⚠️ En SQL, l'égalité s'écrit avec **un seul** `=` (pas `==` comme en Python).

In [ ]:
requete = """
    SELECT IDT_AC, SLD_CTR, COD_ECV_CTR
    FROM CTR
    WHERE COD_ECV_CTR = 4        -- égalité : un seul signe =  (4 = code "Clôturé")
"""
clotures = pd.read_sql(requete, conn)
print(f"Contrats clôturés : {len(clotures)}")

soldes_negatifs = pd.read_sql("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR < 0", conn)
print(f"Soldes négatifs : {len(soldes_negatifs)}")

### ✏️ Exercice 3 — Filtrer les gros soldes

In [ ]:
# TODO : écrire une requête WHERE qui récupère IDT_AC et SLD_CTR pour les contrats
#        avec SLD_CTR > 20000

# --- votre code ---

### ✅ Correction Exercice 3

In [ ]:
gros_soldes = pd.read_sql("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR > 20000", conn)
print(gros_soldes)

🔗 **Pour aller plus loin :**
[W3Schools — SQL WHERE](https://www.w3schools.com/sql/sql_where.asp)

<a id="s7"></a>
## 7. ORDER BY

`ORDER BY colonne` trie le résultat — `ASC` (croissant, par défaut) ou `DESC`
(décroissant). On peut trier sur plusieurs colonnes, dans un ordre de priorité.

In [ ]:
requete = """
    SELECT IDT_AC, SLD_CTR
    FROM CTR
    ORDER BY SLD_CTR DESC   -- DESC : décroissant (les plus gros soldes en premier)
    LIMIT 5                  -- voir Section 13 pour LIMIT
"""
top5 = pd.read_sql(requete, conn)
print(top5)

# Tri sur PLUSIEURS colonnes : d'abord COD_ECV_CTR, puis SLD_CTR décroissant DANS chaque statut
multi_tri = pd.read_sql(
    "SELECT IDT_AC, COD_ECV_CTR, SLD_CTR FROM CTR ORDER BY COD_ECV_CTR ASC, SLD_CTR DESC LIMIT 8",
    conn
)
print(multi_tri)

### ✏️ Exercice 4 — Les 5 plus petits soldes

In [ ]:
# TODO : écrire une requête qui trie CTR par SLD_CTR croissant et garde les 5 premiers

# --- votre code ---

### ✅ Correction Exercice 4

In [ ]:
requete = "SELECT IDT_AC, SLD_CTR FROM CTR ORDER BY SLD_CTR ASC LIMIT 5"
resultat = pd.read_sql(requete, conn)
print(resultat)

🔗 **Pour aller plus loin :**
[W3Schools — SQL ORDER BY](https://www.w3schools.com/sql/sql_orderby.asp)

<a id="s8"></a>
## 8. AND, OR, NOT

Combiner plusieurs conditions dans un `WHERE` :

- `AND` : **toutes** les conditions doivent être vraies
- `OR` : **au moins une** condition doit être vraie
- `NOT` : inverse une condition

💡 Contrairement à Pandas (qui exige `&`/`|` avec des parenthèses obligatoires), en
SQL on écrit `AND`/`OR` en toutes lettres, sans parenthèses obligatoires (mais
recommandées dès que plusieurs `AND`/`OR` se mélangent, pour la lisibilité et éviter
les erreurs de priorité).

In [ ]:
# AND : les DEUX conditions doivent être vraies
requete_and = """
    SELECT IDT_AC, SLD_CTR, COD_DEV
    FROM CTR
    WHERE COD_ECV_CTR = 4 AND COD_DEV = 'EUR'    -- clôturé ET en EUR
"""
print("AND :", len(pd.read_sql(requete_and, conn)), "lignes")

# OR : AU MOINS UNE condition vraie
requete_or = """
    SELECT IDT_AC, COD_ECV_CTR
    FROM CTR
    WHERE COD_ECV_CTR = 4 OR COD_ECV_CTR = 6      -- clôturé OU résilié
"""
print("OR  :", len(pd.read_sql(requete_or, conn)), "lignes")

# Combiner AND/OR : PARENTHÈSES fortement recommandées pour la priorité
requete_combinee = """
    SELECT IDT_AC, SLD_CTR, COD_ECV_CTR
    FROM CTR
    WHERE (COD_ECV_CTR = 4 OR COD_ECV_CTR = 6) AND SLD_CTR < 0   -- (clôturé OU résilié) ET solde négatif
"""
print("Combiné :", len(pd.read_sql(requete_combinee, conn)), "lignes")

# NOT : inverse une condition
requete_not = "SELECT IDT_AC FROM CTR WHERE NOT COD_DEV = 'EUR'"    # tout SAUF EUR
print("NOT :", len(pd.read_sql(requete_not, conn)), "lignes")

### ✏️ Exercice 5 — Comptes à risque en devise étrangère

In [ ]:
# TODO : écrire une requête qui récupère les contrats avec SLD_CTR < 0
#        ET COD_DEV différent de 'EUR' (utiliser NOT ou !=)

# --- votre code ---

### ✅ Correction Exercice 5

In [ ]:
requete = """
    SELECT IDT_AC, SLD_CTR, COD_DEV
    FROM CTR
    WHERE SLD_CTR < 0 AND COD_DEV != 'EUR'
"""
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL AND, OR and NOT](https://www.w3schools.com/sql/sql_and_or.asp)

<a id="s9"></a>
## 9. INSERT INTO

`INSERT INTO table (colonnes) VALUES (valeurs)` ajoute une nouvelle ligne. Pour ne
pas perturber les 5 tables Beobank (utilisées dans tout le reste du notebook), on
travaille sur une table de démonstration dédiée : `notes_conseiller`.

In [ ]:
# .execute(sql) : exécute une instruction SQL qui ne renvoie pas de résultat (CREATE, INSERT, UPDATE, DELETE)
conn.execute("DROP TABLE IF EXISTS notes_conseiller")   # repart propre si la cellule est relancée

conn.execute("""
    CREATE TABLE notes_conseiller (
        IDT_AC      TEXT,
        NOTE        INTEGER,
        COMMENTAIRE TEXT
    )
""")

# INSERT INTO table (colonnes) VALUES (?, ?, ?) -- les "?" sont des PARAMÈTRES,
# remplacés par le tuple donné en 2e argument (évite les injections SQL, bonne pratique)
conn.execute(
    "INSERT INTO notes_conseiller (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    ("AC00001", 8, "Bon client")
)

# .executemany(sql, liste_de_tuples) : insère PLUSIEURS lignes d'un coup
conn.executemany(
    "INSERT INTO notes_conseiller (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    [("AC00002", 5, "Client moyen"), ("AC00003", 9, "Excellent client")]
)
conn.commit()    # .commit() : valide définitivement les changements dans la base

print(pd.read_sql("SELECT * FROM notes_conseiller", conn))

### ✏️ Exercice 6 — Ajouter des notes

In [ ]:
# TODO 1 : insérer la ligne ("AC00004", 3, "A recontacter") dans notes_conseiller
# TODO 2 : avec executemany(), insérer 2 lignes supplémentaires de votre choix
# TODO 3 : vérifier avec un SELECT * FROM notes_conseiller

# --- votre code ---

### ✅ Correction Exercice 6

In [ ]:
conn.execute(
    "INSERT INTO notes_conseiller (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    ("AC00004", 3, "A recontacter")
)
conn.executemany(
    "INSERT INTO notes_conseiller (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    [("AC00005", 7, "RAS"), ("AC00006", 2, "Litige en cours")]
)
conn.commit()
print(pd.read_sql("SELECT * FROM notes_conseiller", conn))

💡 **Repère Vertica :** exactement la même instruction `INSERT INTO ... VALUES (?, ?, ?)`
fonctionne sur Vertica, à ceci près que le placeholder s'écrit `%s` (voir Section 3bis).
Pour charger un **gros volume** de lignes (des milliers/millions), Vertica propose en
plus la commande `COPY table FROM 'fichier.csv' DELIMITER ','` — bien plus rapide
qu'une boucle d'`INSERT`, car pensée pour le chargement en masse (voir Section 29).

🔗 **Pour aller plus loin :**
[W3Schools — SQL INSERT INTO](https://www.w3schools.com/sql/sql_insert.asp)

<a id="s10"></a>
## 10. Valeurs NULL

`NULL` représente une valeur **absente/inconnue** — l'équivalent SQL du `NaN` Pandas
ou du `None` Python. **On ne peut JAMAIS tester `NULL` avec `=`** (`colonne = NULL`
ne fonctionne pas et ne renvoie jamais rien) : il faut `IS NULL` / `IS NOT NULL`.

In [ ]:
# IS NULL : lignes où la colonne est manquante
sans_solde = pd.read_sql("SELECT IDT_AC FROM CTR WHERE SLD_CTR IS NULL", conn)
print(f"Contrats sans solde connu : {len(sans_solde)}")

# IS NOT NULL : lignes où la colonne EST renseignée
avec_solde = pd.read_sql("SELECT IDT_AC FROM CTR WHERE SLD_CTR IS NOT NULL", conn)
print(f"Contrats avec solde connu : {len(avec_solde)}")

# Piège : ceci ne renverra JAMAIS rien, même s'il y a des NULL (= ne fonctionne pas avec NULL)
piege = pd.read_sql("SELECT IDT_AC FROM CTR WHERE SLD_CTR = NULL", conn)
print(f"Avec '= NULL' (piège) : {len(piege)} lignes -- toujours 0, quelle que soit la donnée")

### ✏️ Exercice 7 — Adresses email manquantes

In [ ]:
# TODO : écrire une requête sur TIE_ADR qui compte les lignes où ADR_EMA est NULL

# --- votre code ---

### ✅ Correction Exercice 7

In [ ]:
manquants = pd.read_sql("SELECT * FROM TIE_ADR WHERE ADR_EMA IS NULL", conn)
print(f"Emails manquants : {len(manquants)}")

🔗 **Pour aller plus loin :**
[W3Schools — SQL NULL Values](https://www.w3schools.com/sql/sql_null_values.asp)

<a id="s11"></a>
## 11. UPDATE

`UPDATE table SET colonne = valeur WHERE condition` modifie des lignes existantes.

⚠️ **Un `UPDATE` SANS `WHERE` modifie TOUTES les lignes de la table** — l'erreur la
plus dangereuse en SQL. Toujours vérifier son `WHERE` avec un `SELECT` d'abord.

In [ ]:
# Toujours vérifier AVANT avec un SELECT : combien de lignes seront concernées ?
a_modifier = pd.read_sql("SELECT * FROM notes_conseiller WHERE IDT_AC = 'AC00001'", conn)
print("Avant UPDATE :\n", a_modifier)

# UPDATE ... SET ... WHERE ... -- paramètre "?" pour la valeur, WHERE pour cibler UNE ligne précise
conn.execute(
    "UPDATE notes_conseiller SET NOTE = 10 WHERE IDT_AC = ?",
    ("AC00001",)
)
conn.commit()

apres = pd.read_sql("SELECT * FROM notes_conseiller WHERE IDT_AC = 'AC00001'", conn)
print("Après UPDATE :\n", apres)

### ✏️ Exercice 8 — Corriger un commentaire

In [ ]:
# TODO 1 : vérifier avec un SELECT la ligne où IDT_AC = 'AC00002'
# TODO 2 : mettre à jour son COMMENTAIRE à "Contacté le 2026-01-15"
# TODO 3 : vérifier le résultat avec un nouveau SELECT

# --- votre code ---

### ✅ Correction Exercice 8

In [ ]:
print(pd.read_sql("SELECT * FROM notes_conseiller WHERE IDT_AC = 'AC00002'", conn))

conn.execute(
    "UPDATE notes_conseiller SET COMMENTAIRE = ? WHERE IDT_AC = ?",
    ("Contacté le 2026-01-15", "AC00002")
)
conn.commit()

print(pd.read_sql("SELECT * FROM notes_conseiller WHERE IDT_AC = 'AC00002'", conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL UPDATE](https://www.w3schools.com/sql/sql_update.asp)

<a id="s12"></a>
## 12. DELETE

`DELETE FROM table WHERE condition` supprime des lignes.

⚠️ **Même danger que `UPDATE`** : un `DELETE FROM table` sans `WHERE` supprime
**TOUTES** les lignes de la table (mais garde la table elle-même — pour la table
ET son contenu, voir `DROP TABLE` en Section 29).

In [ ]:
avant = pd.read_sql("SELECT * FROM notes_conseiller", conn)
print(f"Avant DELETE : {len(avant)} lignes")

conn.execute("DELETE FROM notes_conseiller WHERE NOTE < 4")   # supprime les lignes avec une mauvaise note
conn.commit()

apres = pd.read_sql("SELECT * FROM notes_conseiller", conn)
print(f"Après DELETE : {len(apres)} lignes")
print(apres)

### ✏️ Exercice 9 — Supprimer une note obsolète

In [ ]:
# TODO : supprimer la ligne où IDT_AC = 'AC00004', puis vérifier avec un SELECT *

# --- votre code ---

### ✅ Correction Exercice 9

In [ ]:
conn.execute("DELETE FROM notes_conseiller WHERE IDT_AC = 'AC00004'")
conn.commit()
print(pd.read_sql("SELECT * FROM notes_conseiller", conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL DELETE](https://www.w3schools.com/sql/sql_delete.asp)

<a id="s13"></a>
## 13. LIMIT (SELECT TOP)

Pour ne récupérer qu'un certain NOMBRE de lignes : **le mot-clé dépend du moteur SQL** !

| Moteur | Syntaxe |
|---|---|
| SQLite, MySQL, PostgreSQL | `SELECT ... LIMIT n` |
| SQL Server, MS Access | `SELECT TOP n ...` |
| Oracle (récent) | `SELECT ... FETCH FIRST n ROWS ONLY` |

Cette formation utilise **SQLite**, donc `LIMIT`. Retenez surtout que **la syntaxe
SQL n'est pas identique à 100 % d'un moteur à l'autre** — la logique reste la même.

In [ ]:
top5 = pd.read_sql("SELECT IDT_AC, SLD_CTR FROM CTR ORDER BY SLD_CTR DESC LIMIT 5", conn)
print(top5)

# LIMIT + OFFSET : sauter les 5 premiers résultats, puis en prendre 5 -> pagination
page2 = pd.read_sql("SELECT IDT_AC, SLD_CTR FROM CTR ORDER BY SLD_CTR DESC LIMIT 5 OFFSET 5", conn)
print(page2)

### ✏️ Exercice 10 — Top 3 des plus gros contrats

In [ ]:
# TODO : récupérer les 3 contrats avec le plus gros MNT_INI (montant initial)

# --- votre code ---

### ✅ Correction Exercice 10

In [ ]:
top3 = pd.read_sql("SELECT IDT_AC, MNT_INI FROM CTR ORDER BY MNT_INI DESC LIMIT 3", conn)
print(top3)

💡 **Repère Vertica :** Vertica supporte **les deux syntaxes**, `LIMIT n` (comme
SQLite/MySQL/PostgreSQL) ET `SELECT TOP n` (comme SQL Server) — `LIMIT ... OFFSET ...`
reste la forme la plus courante et la plus portable, à privilégier par cohérence.

🔗 **Pour aller plus loin :**
[W3Schools — SQL SELECT TOP / LIMIT](https://www.w3schools.com/sql/sql_top.asp)

<a id="s14"></a>
## 14. Fonctions d'agrégation (COUNT, SUM, AVG, MIN, MAX)

Elles résument une colonne en **UNE seule valeur** — équivalent SQL de `len()`,
`.sum()`, `.mean()`, `.min()`, `.max()` en Pandas.

In [ ]:
requete = """
    SELECT
        COUNT(*)        AS nb_contrats,     -- COUNT(*) : nombre TOTAL de lignes
        COUNT(SLD_CTR)   AS nb_soldes_connus, -- COUNT(colonne) : nombre de lignes où la colonne N'EST PAS NULL
        SUM(SLD_CTR)     AS solde_total,      -- SUM() : somme (ignore les NULL automatiquement)
        AVG(SLD_CTR)     AS solde_moyen,      -- AVG() : moyenne (ignore les NULL)
        MIN(SLD_CTR)     AS solde_min,        -- MIN() : plus petite valeur
        MAX(SLD_CTR)     AS solde_max         -- MAX() : plus grande valeur
    FROM CTR
"""
stats = pd.read_sql(requete, conn)
print(stats)

### ✏️ Exercice 11 — Statistiques sur les transactions

In [ ]:
# TODO : écrire une requête sur TXN_X_CTR qui calcule COUNT(*), SUM(MNT_MVT),
#        AVG(MNT_MVT), MIN(MNT_MVT), MAX(MNT_MVT) avec des alias clairs

# --- votre code ---

### ✅ Correction Exercice 11

In [ ]:
requete = """
    SELECT
        COUNT(*)     AS nb_txn,
        SUM(MNT_MVT)  AS montant_total,
        AVG(MNT_MVT)  AS montant_moyen,
        MIN(MNT_MVT)  AS montant_min,
        MAX(MNT_MVT)  AS montant_max
    FROM TXN_X_CTR
"""
print(pd.read_sql(requete, conn).round(2))

🔗 **Pour aller plus loin :**
[W3Schools — SQL Aggregate Functions](https://www.w3schools.com/sql/sql_aggregate_functions.asp) ·
[W3Schools — COUNT()](https://www.w3schools.com/sql/sql_count.asp) ·
[W3Schools — SUM()](https://www.w3schools.com/sql/sql_sum.asp) ·
[W3Schools — AVG()](https://www.w3schools.com/sql/sql_avg.asp)

<a id="s15"></a>
## 15. LIKE et les jokers (wildcards)

`LIKE` recherche un motif TEXTE, avec deux jokers :
- `%` : remplace **n'importe quelle suite de caractères** (même vide)
- `_` : remplace **exactement UN caractère**

In [ ]:
# '%SEPA%' : contient "SEPA" N'IMPORTE OÙ dans le texte
sepa = pd.read_sql("SELECT LIB_OPE_INL_1 FROM TXN_X_CTR WHERE LIB_OPE_INL_1 LIKE '%SEPA%' LIMIT 5", conn)
print(sepa)

# 'AC0000_' : "AC0000" suivi d'EXACTEMENT un caractère -> AC00001 à AC00009
motif_precis = pd.read_sql("SELECT IDT_AC FROM CTR WHERE IDT_AC LIKE 'AC0000_' LIMIT 5", conn)
print(motif_precis)

# NOT LIKE : l'inverse -- ne contient PAS le motif
sans_sepa = pd.read_sql("SELECT COUNT(*) AS nb FROM TXN_X_CTR WHERE LIB_OPE_INL_1 NOT LIKE '%SEPA%'", conn)
print(sans_sepa)

### ✏️ Exercice 12 — Rechercher les virements

In [ ]:
# TODO : rechercher dans LIB_OPE_INL_1 les libellés contenant "Domiciliation" (LIKE '%...%')

# --- votre code ---

### ✅ Correction Exercice 12

In [ ]:
requete = "SELECT LIB_OPE_INL_1 FROM TXN_X_CTR WHERE LIB_OPE_INL_1 LIKE '%Domiciliation%' LIMIT 5"
print(pd.read_sql(requete, conn))

💡 **Repère Vertica :** `LIKE` est **sensible à la casse** sur Vertica (contrairement à
SQLite, où `LIKE` ignore déjà la casse sur l'ASCII par défaut). Pour une recherche
texte insensible à la casse sur Vertica, utiliser `ILIKE` à la place de `LIKE`
(même syntaxe, mêmes jokers `%` et `_`).

🔗 **Pour aller plus loin :**
[W3Schools — SQL LIKE](https://www.w3schools.com/sql/sql_like.asp) ·
[W3Schools — SQL Wildcards](https://www.w3schools.com/sql/sql_wildcards.asp)

<a id="s16"></a>
## 16. IN

`colonne IN (v1, v2, v3)` : raccourci pour `colonne = v1 OR colonne = v2 OR colonne = v3`
— équivalent SQL du `.isin()` Pandas.

In [ ]:
requete = """
    SELECT IDT_AC, COD_ECV_CTR
    FROM CTR
    WHERE COD_ECV_CTR IN (4, 6)    -- clôturé (4) OU résilié (6) -- bien plus lisible qu'un OR répété
"""
inactifs = pd.read_sql(requete, conn)
print(f"Contrats inactifs : {len(inactifs)}")

# NOT IN : l'inverse -- tout SAUF ces valeurs
actifs = pd.read_sql("SELECT IDT_AC FROM CTR WHERE COD_ECV_CTR NOT IN (4, 6)", conn)
print(f"Contrats actifs : {len(actifs)}")

### ✏️ Exercice 13 — Devises européennes courantes

In [ ]:
# TODO : écrire une requête WHERE COD_DEV IN ('EUR', 'GBP', 'CHF')

# --- votre code ---

### ✅ Correction Exercice 13

In [ ]:
requete = "SELECT IDT_AC, COD_DEV FROM CTR WHERE COD_DEV IN ('EUR', 'GBP', 'CHF')"
print(pd.read_sql(requete, conn).head())

🔗 **Pour aller plus loin :**
[W3Schools — SQL IN](https://www.w3schools.com/sql/sql_in.asp)

<a id="s17"></a>
## 17. BETWEEN

`colonne BETWEEN debut AND fin` : filtre une **plage de valeurs**, les deux bornes
étant **incluses**. Fonctionne aussi sur des dates et du texte.

In [ ]:
requete = """
    SELECT IDT_AC, SLD_CTR
    FROM CTR
    WHERE SLD_CTR BETWEEN 0 AND 5000    -- équivalent à : SLD_CTR >= 0 AND SLD_CTR <= 5000
"""
tranche = pd.read_sql(requete, conn)
print(f"Soldes entre 0 et 5000 : {len(tranche)}")

# BETWEEN sur des dates (les dates sont stockées en texte ISO -> comparaison alphabétique = chronologique)
periode = pd.read_sql(
    "SELECT IDT_AC, DAT_OUV_CTR FROM CTR WHERE DAT_OUV_CTR BETWEEN '2024-01-01' AND '2024-12-31'",
    conn
)
print(f"Contrats ouverts en 2024 : {len(periode)}")

### ✏️ Exercice 14 — Montants initiaux moyens

In [ ]:
# TODO : filtrer les contrats avec MNT_INI BETWEEN 5000 AND 20000

# --- votre code ---

### ✅ Correction Exercice 14

In [ ]:
requete = "SELECT IDT_AC, MNT_INI FROM CTR WHERE MNT_INI BETWEEN 5000 AND 20000"
print(pd.read_sql(requete, conn).head())

🔗 **Pour aller plus loin :**
[W3Schools — SQL BETWEEN](https://www.w3schools.com/sql/sql_between.asp)

<a id="s18"></a>
## 18. Alias (AS)

`AS` renomme temporairement une colonne ou une table dans le résultat — pour un
nom plus lisible, ou pour raccourcir un nom de table répété plusieurs fois
(indispensable dans les jointures, Sections 19-22).

In [ ]:
requete = """
    SELECT
        IDT_AC   AS compte,      -- alias sur une colonne : renomme dans l'affichage
        SLD_CTR  AS solde_eur
    FROM CTR AS c                 -- alias sur une table : "c" devient un raccourci pour CTR
    WHERE c.SLD_CTR > 0           -- on peut ensuite préfixer les colonnes avec l'alias
    LIMIT 3
"""
print(pd.read_sql(requete, conn))

# Le mot-clé AS est optionnel (mais recommandé pour la lisibilité)
sans_as = pd.read_sql("SELECT IDT_AC compte, SLD_CTR solde FROM CTR LIMIT 2", conn)
print(sans_as)

### ✏️ Exercice 15 — Renommer pour un rapport

In [ ]:
# TODO : sélectionner IDT_AC AS "Numero_Compte" et SLD_CTR AS "Solde_EUR" depuis CTR, LIMIT 5

# --- votre code ---

### ✅ Correction Exercice 15

In [ ]:
requete = 'SELECT IDT_AC AS Numero_Compte, SLD_CTR AS Solde_EUR FROM CTR LIMIT 5'
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL Aliases](https://www.w3schools.com/sql/sql_alias.asp)

<a id="s19"></a>
## 19. Jointures — INNER JOIN

Une **jointure** combine les colonnes de PLUSIEURS tables, en les reliant par une
colonne commune (une clé). `INNER JOIN` (le plus courant) ne garde que les lignes
qui ont une correspondance **dans les deux tables** — équivalent SQL de
`pd.merge(..., how="inner")`.

In [ ]:
requete = """
    SELECT c.IDT_AC, c.SLD_CTR, x.IDT_PI       -- colonnes préfixées par l'alias de leur table
    FROM CTR AS c
    INNER JOIN TIE_X_CTR AS x                   -- table à joindre
        ON c.IDT_AC = x.IDT_AC                   -- ON : la condition de jointure (clé commune)
    LIMIT 5
"""
jointure = pd.read_sql(requete, conn)
print(jointure)

# Jointure à 3 tables : CTR + TIE_X_CTR + TIE -- chaque JOIN ajoute une table de plus
requete_3_tables = """
    SELECT c.IDT_AC, c.SLD_CTR, t.COD_LNG_CTR, t.COD_TYP_TIE
    FROM CTR AS c
    INNER JOIN TIE_X_CTR AS x ON c.IDT_AC = x.IDT_AC
    INNER JOIN TIE AS t       ON x.IDT_PI = t.IDT_PI
    LIMIT 5
"""
print(pd.read_sql(requete_3_tables, conn))

### ✏️ Exercice 16 — Contrats avec la ville du client

In [ ]:
# TODO : joindre CTR (c) + TIE_X_CTR (x) sur IDT_AC, + TIE_ADR (a) sur IDT_PI
#        pour afficher IDT_AC, SLD_CTR, NOM_VIL -- LIMIT 5

# --- votre code ---

### ✅ Correction Exercice 16

In [ ]:
requete = """
    SELECT c.IDT_AC, c.SLD_CTR, a.NOM_VIL
    FROM CTR AS c
    INNER JOIN TIE_X_CTR AS x ON c.IDT_AC = x.IDT_AC
    INNER JOIN TIE_ADR AS a   ON x.IDT_PI = a.IDT_PI
    LIMIT 5
"""
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL Joins (vue d'ensemble)](https://www.w3schools.com/sql/sql_join.asp) ·
[W3Schools — SQL INNER JOIN](https://www.w3schools.com/sql/sql_join_inner.asp)

<a id="s20"></a>
## 20. LEFT JOIN

`LEFT JOIN` garde **TOUTES** les lignes de la table de GAUCHE, même sans
correspondance dans la table de droite (colonnes remplies avec `NULL` dans ce cas)
— équivalent SQL de `pd.merge(..., how="left")`. C'est LA jointure la plus utilisée
en pratique : elle ne fait jamais "disparaître" de lignes de la table principale.

In [ ]:
# Combien de contrats N'ONT AUCUNE transaction ? -> LEFT JOIN + IS NULL sur la table de droite
requete = """
    SELECT c.IDT_AC, t.NUM_ORD_MVT_CPB
    FROM CTR AS c
    LEFT JOIN TXN_X_CTR AS t ON c.IDT_AC = t.IDT_AC
    WHERE t.NUM_ORD_MVT_CPB IS NULL    -- aucune ligne correspondante trouvée à droite -> NULL
"""
sans_transaction = pd.read_sql(requete, conn)
print(f"Contrats sans transaction : {len(sans_transaction)}")

# Comparaison : avec INNER JOIN, ces contrats disparaîtraient complètement du résultat
requete_inner = """
    SELECT COUNT(DISTINCT c.IDT_AC) AS nb
    FROM CTR AS c
    INNER JOIN TXN_X_CTR AS t ON c.IDT_AC = t.IDT_AC
"""
print("Contrats AVEC au moins 1 transaction (INNER JOIN) :", pd.read_sql(requete_inner, conn)["nb"][0])

### ✏️ Exercice 17 — Clients sans adresse email

In [ ]:
# TODO : LEFT JOIN TIE (t) avec TIE_ADR (a) sur IDT_PI, puis filtrer a.ADR_EMA IS NULL OR a.ADR_EMA = ''

# --- votre code ---

### ✅ Correction Exercice 17

In [ ]:
requete = """
    SELECT t.IDT_PI, a.ADR_EMA
    FROM TIE AS t
    LEFT JOIN TIE_ADR AS a ON t.IDT_PI = a.IDT_PI
    WHERE a.ADR_EMA IS NULL OR a.ADR_EMA = ''
"""
print(pd.read_sql(requete, conn).head())

🔗 **Pour aller plus loin :**
[W3Schools — SQL LEFT JOIN](https://www.w3schools.com/sql/sql_join_left.asp)

<a id="s21"></a>
## 21. RIGHT JOIN et FULL JOIN

- `RIGHT JOIN` : symétrique du `LEFT JOIN` — garde TOUTES les lignes de la table de
  DROITE. `A RIGHT JOIN B` донне le même résultat que `B LEFT JOIN A` (tables inversées).
- `FULL JOIN` (ou `FULL OUTER JOIN`) : garde TOUTES les lignes des DEUX tables,
  qu'il y ait une correspondance ou non.

⚠️ **Historiquement, SQLite ne supportait NI `RIGHT JOIN` NI `FULL JOIN`** (seulement
`LEFT JOIN`) — ils ont été ajoutés en version 3.39 (2022). Certains moteurs plus
anciens (vieilles versions MySQL...) ne les supportent toujours pas : dans ce cas,
on simule un `RIGHT JOIN A,B` avec `LEFT JOIN B,A` (tables inversées).

In [ ]:
print("Version de SQLite utilisée :", sqlite3.sqlite_version)   # RIGHT/FULL JOIN nécessitent SQLite >= 3.39

# RIGHT JOIN : garde TOUTES les lignes de TXN_X_CTR (droite), même sans contrat correspondant dans CTR
requete_right = """
    SELECT c.IDT_AC AS compte_ctr, t.IDT_AC AS compte_txn
    FROM CTR AS c
    RIGHT JOIN TXN_X_CTR AS t ON c.IDT_AC = t.IDT_AC
    LIMIT 5
"""
print(pd.read_sql(requete_right, conn))

# FULL JOIN : toutes les lignes des DEUX côtés (utile pour un contrôle de cohérence complet)
requete_full = """
    SELECT c.IDT_AC AS compte_ctr, t.IDT_AC AS compte_txn
    FROM CTR AS c
    FULL JOIN TXN_X_CTR AS t ON c.IDT_AC = t.IDT_AC
    WHERE c.IDT_AC IS NULL OR t.IDT_AC IS NULL   -- ne garder QUE les lignes sans correspondance (anomalies)
"""
print(f"Lignes orphelines (l'un des 2 côtés manquant) : {len(pd.read_sql(requete_full, conn))}")

### ✏️ Exercice 18 — Vérifier la cohérence TIE / TIE_X_CTR

In [ ]:
# TODO : écrire un RIGHT JOIN de TIE (gauche) vers TIE_X_CTR (droite) sur IDT_PI,
#        et compter les lignes où TIE.IDT_PI est NULL (un lien TIE_X_CTR sans client connu -> anomalie)

# --- votre code ---

### ✅ Correction Exercice 18

In [ ]:
requete = """
    SELECT t.IDT_PI AS client, x.IDT_AC
    FROM TIE AS t
    RIGHT JOIN TIE_X_CTR AS x ON t.IDT_PI = x.IDT_PI
    WHERE t.IDT_PI IS NULL
"""
print(f"Anomalies : {len(pd.read_sql(requete, conn))}")

🔗 **Pour aller plus loin :**
[W3Schools — SQL RIGHT JOIN](https://www.w3schools.com/sql/sql_join_right.asp) ·
[W3Schools — SQL FULL OUTER JOIN](https://www.w3schools.com/sql/sql_join_full.asp)

<a id="s22"></a>
## 22. SELF JOIN

Une **jointure d'une table avec elle-même** — utile pour comparer des lignes ENTRE
ELLES au sein de la MÊME table. Toujours utiliser 2 alias différents pour distinguer
les deux "copies" de la table dans la requête.

**Cas d'usage Beobank :** trouver les **co-titulaires** — deux clients différents
liés au MÊME contrat dans `TIE_X_CTR`.

In [ ]:
requete = """
    SELECT a.IDT_AC, a.IDT_PI AS client_1, b.IDT_PI AS client_2
    FROM TIE_X_CTR AS a
    INNER JOIN TIE_X_CTR AS b            -- la MÊME table, avec un 2e alias "b"
        ON a.IDT_AC = b.IDT_AC            -- même contrat...
        AND a.IDT_PI < b.IDT_PI            -- ...mais client différent (< évite les doublons et l'auto-comparaison)
    LIMIT 10
"""
co_titulaires = pd.read_sql(requete, conn)
print(f"Paires de co-titulaires trouvées : {len(co_titulaires)}")
print(co_titulaires)

### ✏️ Exercice 19 — Compter les contrats multi-titulaires

In [ ]:
# TODO : à partir de la requête ci-dessus (co-titulaires), compter le nombre de
#        contrats DISTINCTS concernés (COUNT(DISTINCT a.IDT_AC))

# --- votre code ---

### ✅ Correction Exercice 19

In [ ]:
requete = """
    SELECT COUNT(DISTINCT a.IDT_AC) AS nb_contrats_multi_titulaires
    FROM TIE_X_CTR AS a
    INNER JOIN TIE_X_CTR AS b
        ON a.IDT_AC = b.IDT_AC AND a.IDT_PI < b.IDT_PI
"""
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL Self Join](https://www.w3schools.com/sql/sql_join_self.asp)

<a id="s23"></a>
## 23. UNION et UNION ALL

`UNION` empile les résultats de **deux `SELECT`** (mêmes colonnes, même ordre, types
compatibles) — équivalent SQL de `pd.concat()`.

- `UNION` : supprime les doublons (un peu plus lent, car il doit comparer les lignes)
- `UNION ALL` : garde tout, doublons compris (plus rapide, à préférer si on SAIT
  qu'il n'y a pas de doublons possibles ou qu'on s'en fiche)

In [ ]:
requete = """
    SELECT IDT_AC FROM CTR WHERE SLD_CTR < 0          -- comptes à solde négatif

    UNION                                               -- empile le 2e SELECT, sans doublons

    SELECT IDT_AC FROM CTR WHERE COD_ECV_CTR IN (4, 6)  -- comptes clôturés ou résiliés
"""
comptes_a_surveiller = pd.read_sql(requete, conn)
print(f"Comptes à surveiller (solde négatif OU inactif) : {len(comptes_a_surveiller)}")

# UNION ALL : garde les doublons (un compte qui remplit les 2 conditions apparaît 2 fois)
requete_all = requete.replace("UNION", "UNION ALL")
print(f"Avec UNION ALL (doublons compris) : {len(pd.read_sql(requete_all, conn))}")

### ✏️ Exercice 20 — Fusionner deux listes de vigilance

In [ ]:
# TODO : UNION de 2 SELECT IDT_AC : ceux avec MNT_INI > 40000, et ceux avec COD_DEV = 'USD'

# --- votre code ---

### ✅ Correction Exercice 20

In [ ]:
requete = """
    SELECT IDT_AC FROM CTR WHERE MNT_INI > 40000
    UNION
    SELECT IDT_AC FROM CTR WHERE COD_DEV = 'USD'
"""
print(f"{len(pd.read_sql(requete, conn))} comptes dans la liste fusionnée")

🔗 **Pour aller plus loin :**
[W3Schools — SQL UNION](https://www.w3schools.com/sql/sql_union.asp)

<a id="s24"></a>
## 24. GROUP BY

`GROUP BY colonne` regroupe les lignes ayant la MÊME valeur, pour calculer une
agrégation (Section 14) **par groupe** — équivalent SQL de `.groupby().agg()` en
Pandas.

In [ ]:
requete = """
    SELECT
        COD_ECV_CTR,               -- la colonne de regroupement
        COUNT(*)     AS nb,         -- une ligne de résultat PAR VALEUR de COD_ECV_CTR
        AVG(SLD_CTR)  AS solde_moyen,
        SUM(SLD_CTR)  AS solde_total
    FROM CTR
    GROUP BY COD_ECV_CTR           -- un groupe par valeur distincte de statut
    ORDER BY nb DESC
"""
rapport = pd.read_sql(requete, conn)
print(rapport.round(2))

# GROUP BY sur PLUSIEURS colonnes : un groupe par COMBINAISON (statut, devise)
requete_multi = """
    SELECT COD_ECV_CTR, COD_DEV, COUNT(*) AS nb
    FROM CTR
    GROUP BY COD_ECV_CTR, COD_DEV
    ORDER BY COD_ECV_CTR, COD_DEV
"""
print(pd.read_sql(requete_multi, conn).head(8))

### ✏️ Exercice 21 — Transactions par compte

In [ ]:
# TODO : GROUP BY IDT_AC sur TXN_X_CTR, calculer COUNT(*) AS nb_txn et SUM(MNT_MVT) AS total
#        trier par total décroissant, LIMIT 10

# --- votre code ---

### ✅ Correction Exercice 21

In [ ]:
requete = """
    SELECT IDT_AC, COUNT(*) AS nb_txn, SUM(MNT_MVT) AS total
    FROM TXN_X_CTR
    GROUP BY IDT_AC
    ORDER BY total DESC
    LIMIT 10
"""
print(pd.read_sql(requete, conn).round(2))

🔗 **Pour aller plus loin :**
[W3Schools — SQL GROUP BY](https://www.w3schools.com/sql/sql_groupby.asp)

<a id="s25"></a>
## 25. HAVING

`HAVING` filtre les GROUPES **après** l'agrégation (`GROUP BY`) — alors que `WHERE`
filtre les LIGNES **avant** l'agrégation. On ne peut PAS écrire `WHERE COUNT(*) > 5`
(le `COUNT` n'existe pas encore au moment où `WHERE` s'applique) : il FAUT `HAVING`.

In [ ]:
requete = """
    SELECT IDT_AC, COUNT(*) AS nb_txn, SUM(MNT_MVT) AS total
    FROM TXN_X_CTR
    GROUP BY IDT_AC
    HAVING COUNT(*) > 10          -- filtre APRÈS le comptage : garder les comptes très actifs
    ORDER BY total DESC
"""
comptes_actifs = pd.read_sql(requete, conn)
print(f"Comptes avec plus de 10 transactions : {len(comptes_actifs)}")
print(comptes_actifs.head())

# WHERE et HAVING peuvent se combiner : WHERE filtre AVANT, HAVING filtre le résultat agrégé
requete_combinee = """
    SELECT IDT_AC, COUNT(*) AS nb_txn_positives
    FROM TXN_X_CTR
    WHERE MNT_MVT > 0              -- filtre les LIGNES : uniquement les mouvements positifs
    GROUP BY IDT_AC
    HAVING COUNT(*) >= 5           -- filtre les GROUPES : au moins 5 mouvements positifs
"""
print(f"\nComptes avec >= 5 mouvements positifs : {len(pd.read_sql(requete_combinee, conn))}")

### ✏️ Exercice 22 — Statuts fréquents

In [ ]:
# TODO : GROUP BY COD_ECV_CTR sur CTR, HAVING COUNT(*) > 20

# --- votre code ---

### ✅ Correction Exercice 22

In [ ]:
requete = """
    SELECT COD_ECV_CTR, COUNT(*) AS nb
    FROM CTR
    GROUP BY COD_ECV_CTR
    HAVING COUNT(*) > 20
"""
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL HAVING](https://www.w3schools.com/sql/sql_having.asp)

<a id="s26"></a>
## 26. CASE

`CASE WHEN ... THEN ... ELSE ... END` — l'équivalent SQL d'un `if/elif/else`,
directement dans une requête. Équivalent de `np.select()` vu au Jour 2.

In [ ]:
requete = """
    SELECT
        IDT_AC,
        SLD_CTR,
        CASE
            WHEN SLD_CTR < 0          THEN 'Critique'    -- 1er cas vérifié
            WHEN SLD_CTR < 5000        THEN 'Faible'       -- sinon, ce cas
            WHEN SLD_CTR < 50000       THEN 'Moyen'         -- sinon, ce cas
            ELSE 'Élevé'                                     -- sinon (aucun cas au-dessus ne correspond)
        END AS segment                                       -- nom de la colonne résultat
    FROM CTR
    LIMIT 8
"""
print(pd.read_sql(requete, conn))

# CASE dans un GROUP BY : compter combien de contrats dans chaque segment, en une seule requête
requete_group = """
    SELECT
        CASE
            WHEN SLD_CTR < 0     THEN 'Critique'
            WHEN SLD_CTR < 5000   THEN 'Faible'
            WHEN SLD_CTR < 50000  THEN 'Moyen'
            ELSE 'Élevé'
        END AS segment,
        COUNT(*) AS nb
    FROM CTR
    GROUP BY segment
"""
print(pd.read_sql(requete_group, conn))

### ✏️ Exercice 23 — Classifier les statuts

In [ ]:
# TODO : écrire un CASE qui transforme COD_ECV_CTR en libellé :
#        1->'Ouvert', 4->'Clôturé', 6->'Résilié', ELSE 'Autre'
#        Afficher IDT_AC, COD_ECV_CTR, le libellé -- LIMIT 8

# --- votre code ---

### ✅ Correction Exercice 23

In [ ]:
requete = """
    SELECT IDT_AC, COD_ECV_CTR,
        CASE COD_ECV_CTR                    -- forme courte : CASE colonne WHEN valeur THEN ...
            WHEN 1 THEN 'Ouvert'
            WHEN 4 THEN 'Clôturé'
            WHEN 6 THEN 'Résilié'
            ELSE 'Autre'
        END AS libelle
    FROM CTR
    LIMIT 8
"""
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL CASE](https://www.w3schools.com/sql/sql_case.asp)

<a id="s27"></a>
## 27. Fonctions NULL (COALESCE)

`COALESCE(valeur1, valeur2, ...)` renvoie la PREMIÈRE valeur **non-NULL** de la
liste — pratique pour remplacer un `NULL` par une valeur par défaut, équivalent
SQL de `.fillna()` en Pandas. (`IFNULL(a, b)` fait la même chose avec 2 arguments,
mais `COALESCE` est standard et accepte plus de deux valeurs.)

In [ ]:
requete = """
    SELECT
        IDT_AC,
        SLD_CTR,
        COALESCE(SLD_CTR, 0) AS solde_sans_null    -- si SLD_CTR est NULL, utilise 0 à la place
    FROM CTR
    WHERE SLD_CTR IS NULL
    LIMIT 5
"""
print(pd.read_sql(requete, conn))

# COALESCE avec plusieurs "de secours" en cascade : essaie SLD_DSP, sinon SLD_CTR, sinon 0
requete_cascade = """
    SELECT IDT_AC, COALESCE(SLD_DSP, SLD_CTR, 0) AS solde_final
    FROM CTR
    LIMIT 5
"""
print(pd.read_sql(requete_cascade, conn))

### ✏️ Exercice 24 — Emails par défaut

In [ ]:
# TODO : sur TIE_ADR, sélectionner IDT_PI et COALESCE(ADR_EMA, 'non renseigné') AS email

# --- votre code ---

### ✅ Correction Exercice 24

In [ ]:
requete = "SELECT IDT_PI, COALESCE(ADR_EMA, 'non renseigné') AS email FROM TIE_ADR LIMIT 8"
print(pd.read_sql(requete, conn))

🔗 **Pour aller plus loin :**
[W3Schools — SQL ISNULL(), NVL(), IFNULL() and COALESCE()](https://www.w3schools.com/sql/sql_isnull.asp)

<a id="s28"></a>
## 28. EXISTS et sous-requêtes

Une **sous-requête** (*subquery*) est un `SELECT` imbriqué DANS une autre requête —
utile quand la condition dépend elle-même d'un calcul SQL.

`EXISTS (sous-requête)` teste si la sous-requête renvoie **AU MOINS UNE ligne**
(renvoie seulement `True`/`False`, très performant car il s'arrête à la 1ère ligne trouvée).

In [ ]:
# EXISTS : clients qui possèdent AU MOINS UN contrat
requete_exists = """
    SELECT t.IDT_PI
    FROM TIE AS t
    WHERE EXISTS (                                    -- sous-requête : existe-t-il...
        SELECT 1 FROM TIE_X_CTR AS x                    -- ...au moins une ligne dans TIE_X_CTR...
        WHERE x.IDT_PI = t.IDT_PI                          -- ...pour CE client précis (référence à la requête externe) ?
    )
    LIMIT 5
"""
print(pd.read_sql(requete_exists, conn))

# Sous-requête dans le WHERE : contrats dont le solde dépasse la MOYENNE de tous les contrats
requete_sous_requete = """
    SELECT IDT_AC, SLD_CTR
    FROM CTR
    WHERE SLD_CTR > (SELECT AVG(SLD_CTR) FROM CTR)   -- la sous-requête calcule d'abord la moyenne
    ORDER BY SLD_CTR DESC
    LIMIT 5
"""
print(pd.read_sql(requete_sous_requete, conn))

### ✏️ Exercice 25 — Contrats sous la moyenne

In [ ]:
# TODO : sélectionner les contrats avec SLD_CTR < (SELECT AVG(SLD_CTR) FROM CTR)

# --- votre code ---

### ✅ Correction Exercice 25

In [ ]:
requete = """
    SELECT IDT_AC, SLD_CTR
    FROM CTR
    WHERE SLD_CTR < (SELECT AVG(SLD_CTR) FROM CTR)
    ORDER BY SLD_CTR
    LIMIT 5
"""
print(pd.read_sql(requete, conn))

💡 Les sous-requêtes sont la base des **CTE** (`WITH ... AS`), approfondies avec des
cas analytiques avancés (fenêtres glissantes, LAG/LEAD) dans `../formateur/Jour3_Formateur.ipynb` Module 1.

🔗 **Pour aller plus loin :**
[W3Schools — SQL EXISTS](https://www.w3schools.com/sql/sql_exists.asp) ·
[W3Schools — SQL ANY, ALL](https://www.w3schools.com/sql/sql_any_all.asp)

<a id="s29"></a>
## 29. CREATE TABLE

`CREATE TABLE` définit une NOUVELLE table : son nom, ses colonnes, et le TYPE de
chaque colonne (`TEXT`, `INTEGER`, `REAL` pour un nombre à virgule...). `DROP TABLE`
supprime une table (et TOUTES ses données) définitivement.

In [ ]:
conn.execute("DROP TABLE IF EXISTS suivi_contact")   # IF EXISTS : n'échoue pas si la table n'existe pas encore

conn.execute("""
    CREATE TABLE suivi_contact (
        IDT_AC       TEXT,           -- TEXT : chaîne de caractères
        DATE_CONTACT TEXT,           -- (SQLite stocke les dates comme du texte ISO)
        STATUT       TEXT
    )
""")

conn.executemany(
    "INSERT INTO suivi_contact (IDT_AC, DATE_CONTACT, STATUT) VALUES (?, ?, ?)",
    [("AC00001", "2026-01-10", "En attente"), ("AC00002", "2026-01-11", "En attente")]
)
conn.commit()

print(pd.read_sql("SELECT * FROM suivi_contact", conn))

### ✏️ Exercice 26 — Créer une table d'alertes

In [ ]:
# TODO 1 : créer une table "alertes_solde" (IDT_AC TEXT, NIVEAU TEXT, DATE_ALERTE TEXT)
# TODO 2 : insérer 2 lignes avec executemany()
# TODO 3 : vérifier avec un SELECT *

# --- votre code ---

### ✅ Correction Exercice 26

In [ ]:
conn.execute("DROP TABLE IF EXISTS alertes_solde")
conn.execute("""
    CREATE TABLE alertes_solde (
        IDT_AC      TEXT,
        NIVEAU      TEXT,
        DATE_ALERTE TEXT
    )
""")
conn.executemany(
    "INSERT INTO alertes_solde (IDT_AC, NIVEAU, DATE_ALERTE) VALUES (?, ?, ?)",
    [("AC00010", "Critique", "2026-01-12"), ("AC00011", "Modéré", "2026-01-12")]
)
conn.commit()
print(pd.read_sql("SELECT * FROM alertes_solde", conn))

💡 **Repère Vertica :**
- Les types de colonnes sont **stricts** et proches du standard SQL : `VARCHAR(n)`,
  `INT`, `NUMERIC(p,s)`, `TIMESTAMP`, `DATE`... (SQLite est beaucoup plus permissif).
- Les tables Vertica vivent dans un **schéma** (`CREATE TABLE mon_schema.ma_table`),
  ce qui permet d'organiser proprement les tables par domaine (ex : `staging.ctr`,
  `beobank.ctr` pour la version validée).
- Pour insérer un GROS volume de données (le cas typique en entreprise, pas ligne par
  ligne) : `COPY mon_schema.ma_table FROM LOCAL 'fichier.csv' DELIMITER ';'` — la
  commande de référence pour charger rapidement un fichier dans Vertica.

🔗 **Pour aller plus loin :**
[W3Schools — SQL CREATE TABLE](https://www.w3schools.com/sql/sql_create_table.asp)

<a id="s30"></a>
## 30. Mini-projet final — rapport analytique complet en SQL

On mobilise `JOIN`, `WHERE`, `CASE`, `GROUP BY`, `HAVING` et `ORDER BY` en une seule
requête, sur les vraies tables Beobank.

### 🏁 Exercice final — Rapport portefeuille par segment et par langue

In [ ]:
# TODO : écrire UNE requête SQL qui :
# 1. Joint CTR + TIE_X_CTR + TIE (INNER JOIN, sur IDT_AC puis IDT_PI)
# 2. Filtre les Personnes Physiques uniquement (COD_TYP_TIE = 1)
# 3. Classe chaque contrat par segment avec CASE (Critique/Faible/Moyen/Élevé, comme Section 26)
# 4. Regroupe par (COD_LNG_CTR, segment) avec GROUP BY
# 5. Calcule COUNT(*) et AVG(SLD_CTR) par groupe
# 6. Garde seulement les groupes avec au moins 3 contrats (HAVING)
# 7. Trie par nombre de contrats décroissant
# 8. Exécute avec pd.read_sql() et affiche le résultat

# --- votre code ---

### ✅ Correction Exercice final

In [ ]:
requete = """
    SELECT
        t.COD_LNG_CTR,
        CASE
            WHEN c.SLD_CTR < 0     THEN 'Critique'
            WHEN c.SLD_CTR < 5000   THEN 'Faible'
            WHEN c.SLD_CTR < 50000  THEN 'Moyen'
            ELSE 'Élevé'
        END AS segment,
        COUNT(*)         AS nb_contrats,
        AVG(c.SLD_CTR)    AS solde_moyen
    FROM CTR AS c
    INNER JOIN TIE_X_CTR AS x ON c.IDT_AC = x.IDT_AC
    INNER JOIN TIE AS t       ON x.IDT_PI = t.IDT_PI
    WHERE t.COD_TYP_TIE = 1                       -- 1 = Personne Physique
    GROUP BY t.COD_LNG_CTR, segment
    HAVING COUNT(*) >= 3
    ORDER BY nb_contrats DESC
"""
rapport_final = pd.read_sql(requete, conn)
print("=" * 55)
print("  RAPPORT PORTEFEUILLE — SEGMENT x LANGUE (PP)")
print("=" * 55)
print(rapport_final.round(2))

<a id="s31"></a>
## 31. Aide-mémoire et liens

| Catégorie | Instructions vues |
|---|---|
| Lecture | `SELECT`, `SELECT DISTINCT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT` |
| Logique | `AND`, `OR`, `NOT`, `IN`, `BETWEEN`, `LIKE`, `IS NULL` / `IS NOT NULL` |
| Écriture | `INSERT INTO`, `UPDATE ... SET ... WHERE`, `DELETE FROM ... WHERE` |
| Définition | `CREATE TABLE`, `DROP TABLE` |
| Agrégation | `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()`, `GROUP BY`, `HAVING` |
| Jointures | `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL JOIN`, self join |
| Combinaison | `UNION`, `UNION ALL` |
| Logique conditionnelle | `CASE WHEN ... THEN ... ELSE ... END` |
| Valeurs par défaut | `COALESCE()` |
| Autres | alias `AS`, sous-requêtes, `EXISTS` |

### Opérateurs SQL courants

| Opérateur | Signification |
|---|---|
| `=`, `!=` (ou `<>`), `>`, `<`, `>=`, `<=` | comparaison |
| `AND`, `OR`, `NOT` | logique |
| `BETWEEN ... AND ...` | plage (bornes incluses) |
| `LIKE`, `%`, `_` | motif texte |
| `IN (...)` | appartenance à une liste |
| `IS NULL`, `IS NOT NULL` | test de valeur manquante |

### Liens pour aller plus loin

- 🎓 [W3Schools — SQL Tutorial (sommaire complet)](https://www.w3schools.com/sql/default.asp)
- 🎓 [W3Schools — SQL Exercises](https://www.w3schools.com/sql/sql_exercises.asp)
- 📘 [Documentation officielle SQLite](https://www.sqlite.org/lang.html)
- 📘 [Python — module sqlite3](https://docs.python.org/3/library/sqlite3.html)

### Suite du parcours

➡️ `../formateur/Jour3_Formateur.ipynb` Modules 1-3 pour le SQL analytique avancé :
CTE (`WITH ... AS`), `PARTITION BY` (équivalent `.transform()`), `ROW_NUMBER()`,
`LAG`/`LEAD` — tout ce qui va au-delà de ce cours SQL de base.